# 03 Models B and C: balanced sampling and group DRO

Needs notebooks 01 and 02 done. Both models use A's grid, the same folds and the worst-group selection rule (CLAUDE.md Decision Log, 2026-11-18). Per-image OOF predictions go to Drive only; the repo gets aggregate JSON. Charts only; no images are opened. GPU: T4 (C runs 32 settings per fold, ~15 min).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, json, subprocess, zipfile, hashlib
os.environ['CRF_ENV'] = 'colab'
REPO = pathlib.Path('/content/robust_child_filtering_model')
URL = 'github.com/isaacmutuma/robust_child_filtering_model.git'
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/robust_child_filtering_model')
os.chdir('/content')
!rm -rf {REPO}
# Private repo: add a Colab secret GITHUB_TOKEN (key icon, left bar; fine-grained token,
# read-only Contents access to this one repo).
if subprocess.run(['git', 'clone', '-q', f'https://{URL}', str(REPO)]).returncode != 0:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    subprocess.run(['git', 'clone', '-q', f'https://x-access-token:{tok}@{URL}', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', f'https://{URL}'], check=True)
!git -C {REPO} log --oneline -1
os.chdir(REPO)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU works, just slower)')

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

def done(stem):
    """True if Drive holds a clean manifest for `stem` and its npz still matches it."""
    m = DRIVE_ROOT / 'results' / 'embeddings' / f'{stem}.json'
    if not m.exists(): return False
    d = json.loads(m.read_text())
    npz = DRIVE_ROOT / d['npz']['drive_path']
    return d['git']['dirty'] is False and npz.exists() and sha256(npz) == d['npz']['sha256']

def save_manifest(rel):
    """Copy a results JSON to Drive after checking it came from clean, committed code."""
    src = REPO / 'results' / rel
    d = json.loads(src.read_text())
    assert d['git']['dirty'] is False, 'code was modified in this runtime'
    dst = DRIVE_ROOT / 'results' / rel; dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text(src.read_text()); print('saved', rel, 'commit', d['git']['commit'][:7])
    return d

## 1. Preconditions

In [ ]:
A = json.loads((DRIVE_ROOT / 'results/models/A.json').read_text())
assert A['git']['dirty'] is False
assert sha256(DRIVE_ROOT / A['oof_file']['drive_path']) == A['oof_file']['sha256'], "A's OOF file changed"
assert done('utkface__clip_vitl14'), 'UTKFace embeddings missing or changed on Drive'
print('A gap', A['report']['tpr_gap_W_minus_B'])

## 2. Model B: race-balanced sampling within each age band (~5 min)

In [ ]:
!python scripts/train_model.py --model B
d = save_manifest('models/B.json')
print([(f['test_fold'], f['chosen'], f['epochs'], round(f['inner_val_worst_group_tpr'], 3)) for f in d['folds']])

## 3. Model C: group DRO over race x age band (~15 min)

In [ ]:
!python scripts/train_model.py --model C
d = save_manifest('models/C.json')
for f in d['folds']:
    print(f['test_fold'], f['chosen'], f['epochs'], 'top DRO weights', f['refit_info'].get('dro_final_weights_top5'))

## 4. Holm-adjusted verdicts vs A (interim until the whole family has run)

In [ ]:
!python scripts/compare_models.py
save_manifest('comparisons.json')

## 5. Aggregate outputs for the repo (JSON only)

In [ ]:
out = DRIVE_ROOT / 'step7_BC_outputs.zip'
files = [DRIVE_ROOT / 'results/models/B.json', DRIVE_ROOT / 'results/models/C.json', DRIVE_ROOT / 'results/comparisons.json']
with zipfile.ZipFile(out, 'w') as z:
    for f in files:
        z.write(f, f.relative_to(DRIVE_ROOT))
print(out, out.stat().st_size, 'bytes, sha256', sha256(out))